# SMVT Top-3 Hit MD Simulation — GPU-Accelerated
**Target**: SMVT (SLC5A6) | **Method**: GAFF2 + AMBER ff14SB + TIP3P
**Compounds**: Naftazone, Phenobarbital, Esketamine (50ns each)
**Debugged**: Removed Google Drive dependency, OpenFF→GAFF2, !cp→subprocess
---
### Setup (run once)

In [ ]:
!pip install -q openmm openmmforcefields pdbfixer rdkit MDAnalysis matplotlib
import openmm as mm
print('OpenMM', mm.__version__)
print('Platform:', mm.Platform.getPlatformByName('CUDA').getName())

### Verify Input Files
Files uploaded via `colab upload` to `/content/`:
- `AF-Q9Y289-F1.pdb` (AlphaFold receptor)
- `NAFTAZONE_docked.pdbqt`, `PHENOBARBITAL_docked.pdbqt`, `ESKETAMINE_docked.pdbqt`
No Google Drive mount needed — everything is local to the Colab VM.

In [ ]:
import os
DATA_DIR = '/content'
os.makedirs(DATA_DIR, exist_ok=True)
print('Files:', sorted([f for f in os.listdir(DATA_DIR) if not f.startswith('.')]))
assert os.path.exists(f'{DATA_DIR}/AF-Q9Y289-F1.pdb'), 'Receptor PDB missing!'
assert os.path.exists(f'{DATA_DIR}/NAFTAZONE_docked.pdbqt'), 'Naftazone PDBQT missing!'
print('All input files ready.')

---
## MD Pipeline

In [ ]:
import numpy as np
import openmm as mm
import openmm.app as app
import openmm.unit as unit
from openmmforcefields.generators import SystemGenerator
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem
import time, json, os, subprocess

# ═══ Config ═══
DATA_DIR = '/content'
OUT_DIR = '/content/trajectories'
os.makedirs(OUT_DIR, exist_ok=True)

RECEPTOR_PDB = f'{DATA_DIR}/AF-Q9Y289-F1.pdb'

TOP3 = [
    ('NAFTAZONE', 'C1=CC=C2C(=C1)C=CC(=O)C2=NNC(=N)N'),
    ('PHENOBARBITAL', 'CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2'),
    ('ESKETAMINE', 'CNC1(C2=CC=CC=C2Cl)CCCCC1=O'),
]

PROD_NS = 50  # 50ns each!
TEMP = 300 * unit.kelvin
PRESSURE = 1.0 * unit.atmosphere
DT = 2.0 * unit.femtoseconds
CUTOFF = 1.0 * unit.nanometer
SAVE_PS = 100

print(f'GPU MD: {len(TOP3)} compounds x {PROD_NS}ns each (GAFF2)')

In [ ]:
def extract_vina_pose(pdbqt_path):
    """Extract first-model coordinates from Vina PDBQT output."""
    pos = []; in_model = False
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith('MODEL'):
                if int(line.split()[1]) == 1: in_model = True; pos = []
                elif in_model: break
            elif line.startswith('ENDMDL') and in_model: break
            elif in_model and (line.startswith('ATOM') or line.startswith('HETATM')):
                try: pos.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                except: continue
    return pos

def prepare_protein(pdb_path, out_path):
    """Fix missing atoms/residues, add hydrogens at pH 7.4."""
    if os.path.exists(out_path): return out_path
    print('  Preparing protein...')
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues(); fixer.findMissingAtoms()
    fixer.addMissingAtoms(); fixer.addMissingHydrogens(pH=7.4)
    with open(out_path, 'w') as f: app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
    return out_path

def prepare_ligand_pdb(smiles, vina_pos, out_path):
    """Generate ligand PDB from SMILES + Vina docking pose."""
    if os.path.exists(out_path): return out_path
    print('  Building ligand from SMILES + Vina pose...')
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    conf = mol.GetConformer()
    for i in range(min(mol.GetNumAtoms(), len(vina_pos))):
        conf.SetAtomPosition(i, (vina_pos[i][0]/10.0, vina_pos[i][1]/10.0, vina_pos[i][2]/10.0))
    Chem.MolToPDBFile(mol, out_path)
    return out_path

def run_gpu_md(name, smiles):
    tag = name; out_dir = f'{OUT_DIR}/{tag}'; os.makedirs(out_dir, exist_ok=True)
    chk_file = f'{out_dir}/{tag}_{PROD_NS}ns.chk'
    if os.path.exists(chk_file):
        print(f'[{name}] Already done. Skipping.'); return
    
    sep = '=' * 50
    print(f'\n{sep}\n[{name}] {PROD_NS}ns GPU MD (GAFF2)\n{sep}')
    t_start = time.time()
    
    # 1. Prepare protein (fix missing atoms, add H)
    prot_clean = f'{out_dir}/protein.pdb'
    prepare_protein(RECEPTOR_PDB, prot_clean)
    
    # 2. Extract Vina docking pose → ligand PDB
    pdbqt = f'{DATA_DIR}/{name}_docked.pdbqt'
    vina_pos = extract_vina_pose(pdbqt)
    print(f'  Vina pose: {len(vina_pos)} atoms')
    lig_pdb = f'{out_dir}/ligand.pdb'
    prepare_ligand_pdb(smiles, vina_pos, lig_pdb)
    
    # 3. SystemGenerator with GAFF2 (NO openff-toolkit dependency)
    print('  Parameterizing with GAFF2...')
    system_gen = SystemGenerator(
        forcefields=['amber/ff14SB.xml', 'amber/tip3p_standard.xml'],
        small_molecule_forcefield='gaff-2.11',
        cache=f'{out_dir}/cache.json'
    )
    
    # 4. Build solvated system (explicit TIP3P water, 0.15M NaCl)
    print('  Building solvated system...')
    prot_pdb = app.PDBFile(prot_clean)
    lig = app.PDBFile(lig_pdb)
    modeller = app.Modeller(prot_pdb.topology, prot_pdb.positions)
    modeller.add(lig.topology, lig.positions)
    modeller.addSolvent(system_gen.forcefield, model='tip3p',
                        padding=0.8*unit.nanometers, ionicStrength=0.15*unit.molar,
                        neutralize=True)
    system = system_gen.create_system(modeller.topology)
    print(f'  System: {system.getNumParticles()} particles')
    
    # 5. Simulation setup (CUDA, mixed precision)
    integrator = mm.LangevinMiddleIntegrator(TEMP, 1.0/unit.picosecond, DT)
    platform = mm.Platform.getPlatformByName('CUDA')
    simulation = app.Simulation(modeller.topology, system, integrator, platform,
                                {'Precision': 'mixed'})
    simulation.context.setPositions(modeller.positions)
    
    # 6. Energy minimization
    print('  Minimizing...')
    simulation.minimizeEnergy(maxIterations=5000)
    
    # 7. NVT heating (50→300K, 30ps total)
    print('  NVT heating (50→300K)...')
    simulation.integrator.setStepSize(1.0*unit.femtoseconds)
    for t in [50, 150, 300]:
        simulation.integrator.setTemperature(t*unit.kelvin)
        simulation.step(int(10*1000/1.0))
    simulation.integrator.setTemperature(TEMP)
    simulation.integrator.setStepSize(DT)
    simulation.step(int(70*1000/2.0))
    
    # 8. NPT equilibration (100ps)
    print('  NPT equil (100ps)...')
    system.addForce(mm.MonteCarloBarostat(PRESSURE, TEMP))
    simulation.context.reinitialize(preserveState=True)
    simulation.step(int(100*1000/2.0))
    
    # 9. Production run
    prod_steps = int(PROD_NS * 1_000_000 / 2.0)
    save_freq = int(SAVE_PS * 1000 / 2.0)
    print(f'  Production: {PROD_NS}ns ({prod_steps:,} steps)...')
    simulation.reporters.append(app.DCDReporter(f'{out_dir}/{tag}_{PROD_NS}ns.dcd', save_freq))
    simulation.reporters.append(app.StateDataReporter(
        f'{out_dir}/{tag}_{PROD_NS}ns.csv', save_freq,
        step=True, time=True, potentialEnergy=True, temperature=True, volume=True, density=True
    ))
    simulation.reporters.append(app.CheckpointReporter(chk_file, save_freq*10))
    
    simulation.step(prod_steps)
    elapsed = (time.time()-t_start)/60
    ns_per_day = PROD_NS / (elapsed/60/24) if elapsed > 0 else 0
    print(f'  [{name}] DONE in {elapsed:.1f}min (~{ns_per_day:.0f} ns/day)')
    
    # Save final frame
    state = simulation.context.getState(getPositions=True)
    with open(f'{out_dir}/{tag}_final.pdb', 'w') as f:
        app.PDBFile.writeFile(simulation.topology, state.getPositions(), f)
    simulation.saveCheckpoint(chk_file)
    
    # Copy to /content/results/ for easy download
    results_dir = '/content/results'
    os.makedirs(results_dir, exist_ok=True)
    subprocess.run(['cp', '-r', out_dir, results_dir + '/'], check=False)

# ═══ Run all 3 ═══
for name, smiles in TOP3:
    try:
        run_gpu_md(name, smiles)
    except Exception as e:
        print(f'[{name}] FAILED: {e}')
        import traceback; traceback.print_exc()

sep = '=' * 50
print(f'\n{sep}')
print('ALL DONE! Results in /content/results/')
for name, _ in TOP3:
    d = f'/content/results/{name}'
    if os.path.exists(d):
        print(f'  {d}: {sorted(os.listdir(d))}')

---
### Analysis (run after MD completes)
Compute RMSD, RMSF, and binding stability metrics.

In [ ]:
import MDAnalysis
import matplotlib.pyplot as plt
import os

for name, _ in TOP3:
    dcd = f'{OUT_DIR}/{name}/{name}_{PROD_NS}ns.dcd'
    pdb = f'{OUT_DIR}/{name}/{name}_final.pdb'
    if not os.path.exists(dcd):
        print(f'[{name}] No trajectory found, skipping.')
        continue
    
    u = MDAnalysis.Universe(pdb, dcd)
    protein = u.select_atoms('protein and backbone')
    
    # RMSD
    from MDAnalysis.analysis import rms
    R = rms.RMSD(u, protein, ref_frame=0).run()
    
    plt.figure(figsize=(10, 4))
    plt.plot(R.results['time'] / 1000, R.results.rmsd[:, 2], label=name)
    plt.xlabel('Time (ns)'); plt.ylabel('RMSD (Å)')
    plt.title(f'{name} — Backbone RMSD')
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}_rmsd.png', dpi=150)
    plt.show()
    print(f'{name}: Mean RMSD = {R.results.rmsd[:, 2].mean():.2f} Å')